# 05 — Dataset Assembly (multi-city, combined)

Cross-checks `01`'s combined, multi-city reconciled points against what
`03`/`04` actually produced on disk (both SVG and TVG graphs must exist
and load cleanly), builds the final training-ready `dataset_index.parquet`,
and audits node/edge-type coverage across the whole dataset before
anything gets trained on it.

**Absorbs the old `05b`'s job too** -- there is no separate "pool per
city, then merge" step anymore. Every city's `01`-`04` output already
lands in ONE combined `interim`/`processed` tree with globally-unique,
city-prefixed `point_id`s (see `01`'s intro), so this single pass over
the combined `reconciled_points.parquet` already covers what `05b` used
to do as a second, separate notebook. No `uid` column is built here
either -- `point_id` already IS the global primary key.

**Deliberately NOT done here:** normalization (fit per-fold/per-repeat,
inside `07`'s training loop -- computing it globally here would leak
information back in) and any fold/split assignment (`01` no longer
assigns `fold_rep{r}` columns at all -- see `01`'s intro; whichever `07`
branch you run decides its own split scheme).

**Requires `04b`** (vocab unification) to have already been run and
swapped in for every city, or the shared embedding tables in `06`/`07`
will be silently misaligned -- this notebook's last QC cell checks for
exactly that before you trust the output.

In [ ]:
# ── Clone/update repo, mount Drive ──────────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
PROCESSED_DIR = Path(paths_cfg["processed_dir"])

SVG_DIR = PROCESSED_DIR / "svg_graphs"
TVG_DIR = PROCESSED_DIR / "tvg_graphs"
INDEX_OUT = PROCESSED_DIR / "dataset_index.parquet"

print(f"Cities: {CITIES}")
print(f"SVG_DIR: {SVG_DIR}")
print(f"TVG_DIR: {TVG_DIR}")
print(f"Output: {INDEX_OUT}")

In [ ]:
import dataset_audit
import torch
import pandas as pd

reconciled = pd.read_parquet(INTERIM_DIR / "reconciled_points.parquet")
all_point_ids = reconciled["point_id"].tolist()
print(f"Points from 01 (all cities combined): {len(all_point_ids)}")
display(reconciled.groupby("city")["label"].agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

In [ ]:
# ── Single pass: check existence + load-validity + accumulate type stats ──
from tqdm.auto import tqdm

status_df, node_stats, edge_stats = dataset_audit.scan_dataset(
    tqdm(all_point_ids, desc="Scanning SVG+TVG pairs"), SVG_DIR, TVG_DIR, torch
)

In [ ]:
# ── Drop report — exactly which points are excluded, and why ────────────
report = dataset_audit.build_drop_report(status_df)

print(f"Total points:     {report['total_points']}")
print(f"Complete pairs:   {report['complete_pairs']}")
print(f"Missing SVG only: {report['missing_svg_only']}")
print(f"Missing TVG only: {report['missing_tvg_only']}")
print(f"Missing both:     {report['missing_both']}")

for label, ids in [("missing_svg_only_ids", report["missing_svg_only_ids"]),
                    ("missing_tvg_only_ids", report["missing_tvg_only_ids"]),
                    ("missing_both_ids", report["missing_both_ids"])]:
    if ids:
        print(f"\n{label} ({len(ids)}): {ids[:15]}{'...' if len(ids) > 15 else ''}")

# Also surface load ERRORS specifically (corrupted files), distinct from
# plain missing files — these need re-running 02/03/04 for that point,
# not just acknowledging a gap.
corrupted = status_df[
    (status_df["svg_error"].notna() & (status_df["svg_error"] != "file not found"))
    | (status_df["tvg_error"].notna() & (status_df["tvg_error"] != "file not found"))
]
if len(corrupted):
    print(f"\n⚠️  {len(corrupted)} points had a LOAD ERROR (not just missing) — "
          f"likely truncated files from an interrupted save. Re-run the relevant "
          f"notebook (02/03/04) for these before trusting the count above:")
    display(corrupted[["point_id", "svg_error", "tvg_error"]])

In [ ]:
# ── Build the final index: only points with BOTH graphs loading cleanly ──
final_df = dataset_audit.filter_complete_points(reconciled, status_df)
print(f"Final dataset size: {len(final_df)} / {len(reconciled)} "
      f"({len(reconciled) - len(final_df)} dropped)")

In [ ]:
# ── QC: class balance overall AND per city, after dropping — may have
#    shifted from what 01 originally reported (e.g. one city losing more
#    points to failed graph construction than another). No fold breakdown
#    here -- 01 no longer assigns fold_rep{r} columns at all. ───────────
print(f"Overall: {(final_df['label']==1).sum()} positive, {(final_df['label']==0).sum()} negative")
print()
display(final_df.groupby("city")["label"].agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

In [ ]:
# ── QC: node/edge-type coverage across the WHOLE dataset — catches a
#    systemic gap (e.g. a node type that never appears anywhere) that
#    per-point QC in 03/04 wouldn't surface. ─────────────────────────────
print("Node type coverage:")
for nt, stats in sorted(node_stats.items()):
    pct = 100 * stats["n_graphs_present"] / len(final_df) if len(final_df) else 0
    flag = "  ⚠️  never appears with content" if stats["n_graphs_present"] == 0 else ""
    print(f"  {nt:20s} total={stats['total_count']:6d}  "
          f"present_in={stats['n_graphs_present']:4d}/{len(final_df)} ({pct:.1f}%){flag}")

print("\nEdge type coverage:")
for ek, stats in sorted(edge_stats.items(), key=lambda kv: str(kv[0])):
    pct = 100 * stats["n_graphs_present"] / len(final_df) if len(final_df) else 0
    flag = "  ⚠️  never appears with content" if stats["n_graphs_present"] == 0 else ""
    print(f"  {str(ek):45s} total={stats['total_count']:6d}  "
          f"present_in={stats['n_graphs_present']:4d}/{len(final_df)} ({pct:.1f}%){flag}")

In [ ]:
# ── Sanity check: confirm highway_type_idx / building type_idx ranges
#    are consistent with a UNIFIED vocab (i.e. 04b was actually run and
#    swapped in for EVERY city) — catches pooling pre-unification graphs
#    before it silently corrupts a training run's shared embedding table.
import random

sample_hw_idx = []
sample_bt_idx = []
for city in CITIES:
    city_ids = final_df.loc[final_df["city"] == city, "point_id"].tolist()
    sample_ids = random.sample(city_ids, min(20, len(city_ids)))
    for pid in sample_ids:
        g = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
        sample_hw_idx.append((city, int(g["incident"].highway_type_idx.item())))
        if g["building"].type_idx.numel():
            sample_bt_idx.append((city, int(g["building"].type_idx[0].item())))

hw_by_city = {}
for city, idx in sample_hw_idx:
    hw_by_city.setdefault(city, []).append(idx)

print("Sampled highway_type_idx range per city:")
for city, idxs in hw_by_city.items():
    print(f"  {city}: {min(idxs)}..{max(idxs)}")
overall_max = max(i for _, i in sample_hw_idx)
print(f"Overall sampled range: 0..{overall_max}")
print()
print("If each city's range looks suspiciously narrow AND non-overlapping")
print("(e.g. one city only ever shows 0-3, another only ever shows 10-13,")
print("never mixing), that's consistent with UNRUN unification -- re-run")
print("04b before trusting this dataset. A unified vocab typically has")
print("every city's samples scattered across the same, shared range.")

In [ ]:
# ── Spot-check a few real graphs directly, across different cities ──────
import random
sample_ids = []
for city in CITIES:
    city_ids = final_df.loc[final_df["city"] == city, "point_id"].tolist()
    sample_ids.extend(random.sample(city_ids, min(2, len(city_ids))))

for pid in sample_ids:
    svg = torch.load(SVG_DIR / f"{pid}.pt", weights_only=False)
    tvg = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
    print(f"\n{pid}:")
    print(f"  SVG node types: {svg.node_types}")
    print(f"  TVG node types: {tvg.node_types}")

In [ ]:
# ── Save the final index ─────────────────────────────────────────────────
final_df.to_parquet(INDEX_OUT, index=False)
print(f"✅ Saved {len(final_df)} rows ({len(CITIES)} cities) to {INDEX_OUT}")
print()
print("Next: 06_models.ipynb — architecture QC forward pass, then 07's")
print("training configuration (batch size, epoch cap, patience, split")
print("scheme, and normalization fitting) still to be decided.")